# A02 — Agente Strands: calcular e comprovar a resposta

![Arquitetura do exercício](assets/code-agent.png)

## O que você vai aprender

O agente usa Amazon Bedrock para decidir qual código executar. A ferramenta `execute_python` chama AgentCore Code Interpreter; a execução Python ocorre na AWS. O SDK Strands coordena a conversa no kernel do notebook.

## Pré-requisitos e sequência

Use uma conta AWS com sessão autorizada em `us-west-2`, Python 3.12 e kernel próprio para este apêndice.
As células instalam as versões validadas e criam os recursos durante a execução. Execute de cima para baixo, incluindo a limpeza.
Uma conta basta; as permissões variam conforme o evento. Estes exemplos são independentes das aulas Santander.

## Como interpretar o resultado

O maior primo entre 9 e 84 é **83**. A resposta só passa se o agente realmente chamou a ferramenta, ela retornou sucesso e a resposta final contém 83. Não basta o modelo acertar sem executar o código.

## Origem e adaptações

Derivado do [notebook oficial](https://github.com/awslabs/agentcore-samples/blob/0d46da96306fa692c9887d0461252bf5aeeee6c3/06-workshops/05-AgentCore-tools/01-Agent-Core-code-interpreter/02-code-execution-with-agent-using-code-interpreter/strands-agent-code-execution-code-interpreter.ipynb), revisão `0d46da96306f`.
Foram adicionados explicações, validações e dependências explícitas; os caminhos dos arquivos foram organizados neste apêndice.
O Browser usa a cadeia padrão de credenciais e nomes exclusivos. O exemplo de cálculo declara Claude Haiku 4.5.
As evidências e diferenças constam no [relatório de validação](VALIDACAO.md).

## Conferir a sessão AWS


In [ ]:
import boto3, os
session = boto3.Session()
REGION = session.region_name or os.environ.get("AWS_DEFAULT_REGION")
assert REGION == "us-west-2", "Este apêndice foi validado em us-west-2; configure a região antes de continuar."
identity = session.client("sts").get_caller_identity()
print({"conta": identity["Account"], "regiao": REGION, "identidade": identity["Arn"]})
# Credenciais vêm da cadeia padrão do boto3: role do ambiente, perfil ou sessão temporária.
# Nenhuma chave, senha ou token deve ser salvo neste notebook.


In [ ]:
%pip install "bedrock-agentcore==1.25.3" "boto3==1.43.124" "strands-agents==1.64.0"


In [ ]:
from bedrock_agentcore.tools.code_interpreter_client import code_session
from strands import Agent, tool
from strands.models import BedrockModel
import json

## 2.System Prompt Definition
Define the behavior and capabilities of the AI assistant. We instruct our assistant to always validate answers through code execution and data based reasoning.

In [ ]:
SYSTEM_PROMPT = """You are a helpful AI assistant that validates all answers through code execution.

VALIDATION PRINCIPLES:
1. When making claims about code, algorithms, or calculations - write code to verify them
2. Use execute_python to test mathematical calculations, algorithms, and logic
3. Create test scripts to validate your understanding before giving answers
4. Always show your work with actual code execution
5. If uncertain, explicitly state limitations and validate what you can

APPROACH:
- If asked about a programming concept, implement it in code to demonstrate
- If asked for calculations, compute them programmatically AND show the code
- If implementing algorithms, include test cases to prove correctness
- Document your validation process for transparency
- The sandbox maintains state between executions, so you can refer to previous results

TOOL AVAILABLE:
- execute_python: Run Python code and see output

RESPONSE FORMAT: The execute_python tool returns a JSON response with:
- sessionId: The sandbox session ID
- id: Request ID
- isError: Boolean indicating if there was an error
- content: Array of content objects with type and text/data
- structuredContent: For code execution, includes stdout, stderr, exitCode, executionTime"""

## 3.Code Execution Tool Definition
Next we define the function as tool that will be used by the Agent as tool, to run code in the code sandbox. We use the @tool decorator to annotate the function as a custom tool for the Agent.

Within an active code interpreter session, you can execute code in supported languages (Python, JavaScript), access libraries based on your dependencies configuration, generate visualizations, and maintain state between executions.

In [ ]:
@tool
def execute_python(code: str, description: str = "") -> str:
    """Execute Python code in the sandbox."""

    if description:
        code = f"# {description}\n{code}"

    print(f"\n Generated Code: {code}")

    with code_session("us-west-2") as code_client:
        response = code_client.invoke("executeCode", {"code": code, "language": "python", "clearContext": False})

    for event in response["stream"]:
        return json.dumps(event["result"])

## 4. Agent Configuration
We create and configure an agent using the Strands SDK.We provide it the system prompt and the tool we defined above to execute generate code

In [ ]:
agent = Agent(model=BedrockModel(model_id="us.anthropic.claude-haiku-4-5-20251001-v1:0", region_name=REGION), tools=[execute_python], system_prompt=SYSTEM_PROMPT, callback_handler=None)

## 5. Query Definition
Define a sample query to test the agent with code execution capabilities

In [ ]:
query = "Tell me the largest random prime number between 1 and 100, which is less than 84 and more that 9"

## 6. Agent Invocation and Response Processing
We invoke the agent with our query and process the agent's response


Note: Async execution requires running in an async environment

In [ ]:
try:
    response_text = ""
    async for event in agent.stream_async(query):
        if "data" in event:
            chunk = event["data"]
            response_text += chunk
            print(chunk, end="")
except Exception as e:
    print(f"Error occurred: {str(e)}")

## Validar o resultado observado

As verificações abaixo falham se o resultado real não corresponder ao exercício.

In [ ]:
assert response_text.strip(), "The agent returned no answer"
assert "83" in response_text, "Expected largest prime below 84: 83"
tool_results = [
    block["toolResult"]
    for message in agent.messages
    for block in message.get("content", [])
    if "toolResult" in block
]
assert tool_results, "The model did not call the AWS tool"
assert all(r.get("status") != "error" for r in tool_results)
for result in tool_results:
    for block in result.get("content", []):
        if "text" in block:
            payload = json.loads(block["text"])
            assert not payload.get("isError"), payload
print("APPENDIX_RESULT_VALIDATED: prime 83 computed using AWS Code Interpreter")